In [ ]:
Домашнее задание
по теме «3.9 Многопоточность в Python»

In [ ]:
Задача 1. Удвоение чисел и получение результата Напишите многопоточный код для обработки чисел из нескольких списков.
Каждое число в списке должно быть умножено на 2, с имитацией задержки 0.2 сек на каждой операции.
Используйте ThreadPoolExecutor и as_completed для управления потоками и отслеживания результатов.

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving test_list_numbers.txt to test_list_numbers.txt


In [ ]:
import time
import re
from concurrent.futures import ThreadPoolExecutor, as_completed


# Парсер строки
def parse_line(line):
    """
    Извлекает числа из строки независимо от формата.
    """
    numbers = re.findall(r"-?\d+", line)
    return [int(x) for x in numbers]


# Загрузка данных из файла
def load_lists(filename):
    lists = []

    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            nums = parse_line(line)

            if nums:
                lists.append(nums)

    return lists


# Обработка одного числа
def process_number(number):
    time.sleep(0.2)
    return number * 2


# Обработка одного списка
def process_list(lst):
    start_time = time.time()

    results = []

    with ThreadPoolExecutor(max_workers=10) as executor:

        futures = [
            executor.submit(process_number, num)
            for num in lst
        ]

        for future in as_completed(futures):
            results.append(future.result())

    end_time = time.time()

    return sum(results), end_time - start_time


# MAIN
filename = "test_list_numbers.txt"

lists = load_lists(filename)

times = []
sums = []

print("Начинаем обработку...\n")

for i, lst in enumerate(lists):

    current_sum, current_time = process_list(lst)

    sums.append(current_sum)
    times.append(current_time)

    print(
        f"Список {i+1}: "
        f"сумма = {current_sum}, "
        f"время = {current_time:.3f} сек"
    )

# самый быстрый список
fastest_index = times.index(min(times))

print("\n====================")
print(f"Самый быстрый список: {fastest_index + 1}")
print(f"Сумма чисел: {sums[fastest_index]}")
print("====================")

Начинаем обработку...

Список 1: сумма = 11090, время = 0.403 сек
Список 2: сумма = 15006, время = 0.403 сек
Список 3: сумма = 11454, время = 0.403 сек
Список 4: сумма = 27746, время = 0.403 сек
Список 5: сумма = 19184, время = 0.403 сек
Список 6: сумма = 16018, время = 0.403 сек
Список 7: сумма = 19446, время = 0.402 сек
Список 8: сумма = 8196, время = 0.202 сек
Список 9: сумма = 14664, время = 0.403 сек
Список 10: сумма = 15548, время = 0.404 сек
Список 11: сумма = 17884, время = 0.402 сек
Список 12: сумма = 5916, время = 0.203 сек
Список 13: сумма = 4878, время = 0.203 сек
Список 14: сумма = 20450, время = 0.402 сек
Список 15: сумма = 7822, время = 0.203 сек

Самый быстрый список: 8
Сумма чисел: 8196


In [ ]:
Задача 2. Поиск и суммирование чисел через цепочку файлов
Для решения используйте многопроцессность с помощью метода pool.starmap.
Вам дан архив с 1000 текстовыми файлами, Архив с файлами 1 (path_8_8.zip).
Задача заключается в том, чтобы написать код, который обрабатывает каждый из этих файлов в многопроцессном режиме.
В каждом файле записан путь к файлу в другом архиве. Архив с файлами 2 (recursive_challenge_8_8.zip).
Ваш код должен следовать этому пути, чтобы найти конечный файл, содержащий число.
Это число необходимо прибавить к глобальному счётчику.

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving recursive_challenge_8_8.zip to recursive_challenge_8_8.zip
Saving path_8_8.zip to path_8_8.zip


In [ ]:
import zipfile
from multiprocessing import Pool


PATH_ZIP = "path_8_8.zip"
DATA_ZIP = "recursive_challenge_8_8.zip"


def process_file(file_name):
    with zipfile.ZipFile(PATH_ZIP) as z1, zipfile.ZipFile(DATA_ZIP) as z2: # Открываем оба архива
        target_path = z1.read(file_name).decode("utf-8").strip()  # Получаем путь к файлу из первого архива
        target_path = target_path.replace("\\", "/")   # Заменяем Windows-разделители на формат ZIP

        number = int(z2.read(target_path).decode("utf-8").strip())   # Читаем число из файла второго архива
        return number

# Получаем список всех txt-файлов из первого архива
with zipfile.ZipFile(PATH_ZIP) as z:
    files = [f for f in z.namelist() if f.endswith(".txt")]

# Создаем пул процессов
with Pool(processes=4) as pool:
    results = pool.map(process_file, files)  # Параллельно обрабатываем все файлы

print("Количество файлов:", len(files))
print("СУММА ЧИСЕЛ:", sum(results))

Количество файлов: 1000
СУММА ЧИСЕЛ: 5152208
